# Azure AI Search 연결/CRUD 테스트

`.env`를 로드해서 Azure AI Search에 연결 → 인덱스 생성(Create) → 문서 업로드/조회(Read) → 검색(Search) → 수정(Update) → 삭제(Delete)까지 한 번씩 해보는 노트북입니다.

`file_storage_test.ipynb`, `blob_storage_test.ipynb`와 목적은 같지만, 여기서는 실제 OCR 캐시가 쓰는 인덱스(`app/services/search_index_service.py`)와 스키마를 헷갈리지 않도록 **별도의 테스트용 인덱스**(`{AZURE_SEARCH_INDEX_NAME}-test`)를 만들어서 그 안에서만 CRUD를 해봅니다.

## 사전 준비
1. 아래 패키지가 설치되어 있어야 합니다.

```bash
pip install azure-search-documents python-dotenv
```
2. 프로젝트 루트(`azure-doc-ai-service/`)의 `.env`에 아래 값이 채워져 있어야 합니다 (없다면 `.env.example`을 복사).

```
AZURE_SEARCH_ENDPOINT=https://<your-search-service>.search.windows.net
AZURE_SEARCH_API_KEY=<your-admin-api-key>
AZURE_SEARCH_INDEX_NAME=ocr-documents
```

`AZURE_SEARCH_API_KEY`는 인덱스 생성/삭제까지 하려면 **관리자(admin) 키**여야 합니다 (쿼리 키로는 Create/Delete가 실패합니다).

In [ ]:
import os
import time
from datetime import datetime, timezone

from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import ResourceExistsError, ResourceNotFoundError
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchFieldDataType, SearchIndex, SimpleField
from dotenv import load_dotenv
from pathlib import Path

# 이 노트북(azure-doc-ai-service/notebooks/)의 부모 폴더(azure-doc-ai-service/)에 있는 .env를 로드
ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"]
API_KEY = os.environ["AZURE_SEARCH_API_KEY"]
BASE_INDEX_NAME = os.environ.get("AZURE_SEARCH_INDEX_NAME", "ocr-documents")

# 실제 서비스가 쓰는 인덱스를 건드리지 않도록 테스트 전용 인덱스를 따로 씀
TEST_INDEX_NAME = f"{BASE_INDEX_NAME}-test"
DOCUMENT_ID = "test-doc-001"

print("ENDPOINT:", ENDPOINT)
print("TEST_INDEX_NAME:", TEST_INDEX_NAME)
print("DOCUMENT_ID:", DOCUMENT_ID)

## 1. 클라이언트 연결

관리자 키로 `SearchIndexClient`를 만들고, 실제로 인증/네트워크가 되는지 확인하기 위해 계정에 있는 인덱스 목록을 한 번 조회합니다.

In [ ]:
credential = AzureKeyCredential(API_KEY)
index_client = SearchIndexClient(endpoint=ENDPOINT, credential=credential)

print("연결 성공, 계정 내 인덱스 목록(최대 10개):")
for i, index in enumerate(index_client.list_indexes()):
    if i >= 10:
        print("  ...")
        break
    print(f"  - {index.name}")

## 2. Create - 테스트 인덱스 생성 + 문서 업로드

실제 서비스 스키마(`_build_index` in `search_index_service.py`)를 단순화한 필드(`id`, `content_hash`, `status`, `created_at`, `updated_at`)로 테스트 인덱스를 만들고, 문서 하나를 업로드합니다.

인덱스는 이미 있으면 그대로 쓰고(`get_index` 성공 시), 문서는 "진짜 새로 생성됐는지"를 확인하려고 업로드 전에 `get_document`로 직접 존재 여부를 체크합니다.

In [ ]:
def build_test_index() -> SearchIndex:
    return SearchIndex(
        name=TEST_INDEX_NAME,
        fields=[
            SimpleField(name="id", type=SearchFieldDataType.String, key=True),
            SimpleField(name="content_hash", type=SearchFieldDataType.String, filterable=True),
            SimpleField(name="status", type=SearchFieldDataType.String, filterable=True),
            SimpleField(name="created_at", type=SearchFieldDataType.DateTimeOffset, filterable=True),
            SimpleField(name="updated_at", type=SearchFieldDataType.DateTimeOffset, filterable=True),
        ],
    )


try:
    index_client.get_index(TEST_INDEX_NAME)
    print(f"인덱스 '{TEST_INDEX_NAME}' 이미 존재함 (그대로 사용)")
except ResourceNotFoundError:
    index_client.create_index(build_test_index())
    print(f"인덱스 '{TEST_INDEX_NAME}' 생성됨")

search_client = SearchClient(endpoint=ENDPOINT, index_name=TEST_INDEX_NAME, credential=credential)


def create_document(document: dict) -> None:
    """id가 이미 있으면 에러를 내도록 만들어서 진짜 '생성'만 테스트한다."""
    try:
        search_client.get_document(key=document["id"])
        raise ResourceExistsError(f"'{document['id']}'가 이미 존재합니다.")
    except ResourceNotFoundError:
        pass
    search_client.upload_documents([document])
    print(f"'{document['id']}' 생성 완료")


now = datetime.now(timezone.utc).isoformat()
sample_document = {
    "id": DOCUMENT_ID,
    "content_hash": "sample-hash-0001",
    "status": "processing",
    "created_at": now,
    "updated_at": now,
}

try:
    create_document(sample_document)
except ResourceExistsError:
    print(f"'{DOCUMENT_ID}'가 이미 존재합니다. 삭제 후 다시 실행하거나 DOCUMENT_ID를 바꾸세요.")

## 3. Read - 문서 조회 + 검색

`get_document`로 방금 만든 문서를 키로 직접 조회하고, `search`로 전체 검색도 한 번 해봅니다.

`get_document`는 즉시 조회지만, `search`는 색인에 반영되는 데 약간의 지연(보통 1~2초)이 있을 수 있어서 재시도를 조금 넣었습니다.

In [ ]:
def read_document(key: str) -> dict:
    return search_client.get_document(key=key)


print(f"'{DOCUMENT_ID}' 조회 결과:")
print(read_document(DOCUMENT_ID))


def search_by_status(status: str, retries: int = 5, delay_seconds: float = 1.0) -> list[dict]:
    for attempt in range(retries):
        results = list(search_client.search(search_text="*", filter=f"status eq '{status}'"))
        if results:
            return results
        time.sleep(delay_seconds)
    return []


print(f"\nstatus='processing' 검색 결과:")
for doc in search_by_status("processing"):
    print(f"  - {doc['id']} (content_hash={doc['content_hash']})")

## 4. Update - 문서 병합 갱신

실제 서비스(`ocr_cache_service.py`)가 쓰는 것과 같은 방식으로, 바뀐 필드만 담아서 `merge_or_upload_documents`를 호출합니다. `id`만 같으면 나머지 필드는 기존 값이 그대로 유지되고, 보낸 필드만 덮어써집니다.

In [ ]:
def update_document(partial_document: dict) -> None:
    search_client.merge_or_upload_documents([partial_document])
    print(f"'{partial_document['id']}' 갱신 완료 (필드: {list(partial_document.keys())})")


update_document({"id": DOCUMENT_ID, "status": "completed", "updated_at": datetime.now(timezone.utc).isoformat()})

print("\n갱신 후 재조회 (content_hash는 그대로, status/updated_at만 바뀌어야 함):")
print(read_document(DOCUMENT_ID))

## 5. Delete - 문서 삭제 (+ 선택: 인덱스 삭제)

테스트에 쓴 문서를 지웁니다. 인덱스 자체를 지우는 셀은 기본적으로 실행 안 되게 주석 처리해뒀습니다 (다른 데이터가 든 인덱스를 실수로 통째로 지우는 걸 막기 위함) — 정말 지우고 싶을 때만 주석을 풀고 실행하세요.

In [ ]:
def delete_document(key: str) -> None:
    result = search_client.delete_documents([{"id": key}])[0]
    if result.succeeded:
        print(f"'{key}' 삭제 완료")
    else:
        print(f"'{key}' 삭제 실패: {result.error_message}")


delete_document(DOCUMENT_ID)

print("\n삭제 후 조회 (ResourceNotFoundError가 나야 정상):")
try:
    read_document(DOCUMENT_ID)
except ResourceNotFoundError:
    print(f"'{DOCUMENT_ID}'가 더 이상 존재하지 않습니다.")

In [ ]:
# 인덱스 자체를 삭제하려면 아래 주석을 풀고 실행하세요 (되돌릴 수 없습니다).
# index_client.delete_index(TEST_INDEX_NAME)
# print(f"인덱스 '{TEST_INDEX_NAME}' 삭제 완료")